In [12]:
# ── Setup ──────────────────────────────────────────────────────────
import os, getpass
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from langchain.tools import tool
from langchain.agents import create_agent

In [13]:
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI key: Your openai key")

os.makedirs("eda_outputs", exist_ok=True)

In [14]:
df = sns.load_dataset("titanic")

In [15]:
def infer_role(series):
    """Return one of: constant, id-like, datetime-like, numeric, categorical."""
    n_non_null = series.notna().sum()
    if series.nunique(dropna=True) <= 1:
        return "constant"
    if series.nunique(dropna=True) == n_non_null:
        return "id-like"
    if pd.api.types.is_datetime64_any_dtype(series):
        return "datetime-like"
    if series.dtype == "object":
        try:
            parsed = pd.to_datetime(series, errors="coerce")
            if parsed.notna().sum() / n_non_null > 0.9:
                return "datetime-like"
        except (ValueError, TypeError):
            pass
    if pd.api.types.is_numeric_dtype(series):
        return "numeric"
    return "categorical"

In [16]:
# ── Profiler ───────────────────────────────────────────────────────
def profile_data(df):
    """Compact, LLM-readable profile of any dataframe."""
    profile = {
        "shape": list(df.shape),
        "duplicate_rows": int(df.duplicated().sum()),
        "memory_mb": round(df.memory_usage(deep=True).sum() / 1e6, 2),
        "columns": {}
    }
    for col in df.columns:
        s = df[col]
        role = infer_role(s)
        non_null = s.dropna()
        n_unique = int(s.nunique(dropna=True))
        null_pct = float(round(s.isna().mean() * 100, 1))
        info = {
            "role": role,
            "dtype": str(s.dtype),
            "null_pct": null_pct,
            "n_unique": n_unique,
            "samples": [str(v) for v in non_null.head(3).tolist()],
            "flags": []
        }
        if role == "numeric" and len(non_null) > 0:
            if pd.api.types.is_bool_dtype(non_null):
                non_null = non_null.astype(int)
            q1, q3 = non_null.quantile(0.25), non_null.quantile(0.75)
            iqr = q3 - q1
            outliers = ((non_null < q1 - 1.5 * iqr) | (non_null > q3 + 1.5 * iqr)).sum()
            info["stats"] = {
                "min": float(non_null.min()),
                "max": float(non_null.max()),
                "mean": round(float(non_null.mean()), 2),
                "median": float(non_null.median()),
                "std": round(float(non_null.std()), 2),
                "outlier_count": int(outliers),
            }
        else:
            top = non_null.value_counts().head(5)
            info["top_values"] = {str(k): int(v) for k, v in top.items()}
        if null_pct > 50:
            info["flags"].append("high_nulls")
        if role == "constant":
            info["flags"].append("constant")
        if role == "numeric" and n_unique <= 10:
            info["flags"].append("maybe_categorical")
        if role == "id-like":
            info["flags"].append("id_like")
        profile["columns"][col] = info
    return profile



In [17]:
# ── Safety scaffold (run once, before the clean tools) ──────────────
work_df = df.copy()      # agent mutates THIS, never the original df
clean_log = []           # running record of every change made

def log_change(msg):
    clean_log.append(msg)
    return msg


@tool
def handle_missing(column: str, strategy: str) -> str:
    """Handle missing values in a column. Call after identifying nulls.

    strategy options:
    - 'median' / 'mean' : fill nulls with that statistic (NUMERIC columns only)
    - 'mode'            : fill nulls with most frequent value (any column)
    - 'drop_rows'       : remove rows where this column is null
    - 'drop_column'     : remove the entire column (use for very high-null columns)

    Args:
        column: exact column name.
        strategy: one of median, mean, mode, drop_rows, drop_column.
    """
    # GUARDRAIL 1 — validate the target exists. Never operate on a missing
    # column; tell the agent what's available so it can correct itself.
    if column not in work_df.columns:
        return f"Column '{column}' not found. Available: {list(work_df.columns)}"

    is_num = pd.api.types.is_numeric_dtype(work_df[column])
    n_null = int(work_df[column].isna().sum())

    # GUARDRAIL 2 — refuse type-invalid operations and explain WHY + what to
    # do instead. This is the key pattern: the tool pushes back, the agent
    # reads the message, and picks a valid strategy on its next turn.
    if strategy in ("mean", "median") and not is_num:
        return (f"Cannot use '{strategy}' on non-numeric column '{column}'. "
                f"Use 'mode' for categorical columns instead.")

    # GUARDRAIL 3 — don't do destructive work for no reason. If there's
    # nothing to fix, say so rather than silently "succeeding".
    if n_null == 0 and strategy not in ("drop_column",):
        return f"Column '{column}' has no missing values — nothing to do."

    # ── The action. Note every branch logs the SPECIFIC value used, so the
    #    change is auditable after the fact, not just "cleaned age". ──
    if strategy == "median":
        val = work_df[column].median()
        work_df[column] = work_df[column].fillna(val)
        return log_change(f"Filled {n_null} nulls in '{column}' with median {val:.2f}")

    if strategy == "mean":
        val = work_df[column].mean()
        work_df[column] = work_df[column].fillna(val)
        return log_change(f"Filled {n_null} nulls in '{column}' with mean {val:.2f}")

    if strategy == "mode":
        val = work_df[column].mode().iloc[0]
        work_df[column] = work_df[column].fillna(val)
        return log_change(f"Filled {n_null} nulls in '{column}' with mode '{val}'")

    if strategy == "drop_rows":
        before = len(work_df)
        work_df.dropna(subset=[column], inplace=True)
        removed = before - len(work_df)
        return log_change(f"Dropped {removed} rows with null '{column}'")

    if strategy == "drop_column":
        work_df.drop(columns=[column], inplace=True)
        return log_change(f"Dropped column '{column}' entirely")

    # GUARDRAIL 4 — unknown strategy. Closed menu, clear error.
    return f"Unknown strategy '{strategy}'. Use median, mean, mode, drop_rows, or drop_column."

In [18]:
# ── Tools ──────────────────────────────────────────────────────────
@tool
def profile_dataset() -> str:
    """Get a high-level profile of the entire dataset: shape, duplicate rows,
    and for every column its role, null %, and key stats or top values.
    Call this FIRST, before any other tool, to decide where to look."""
    p = profile_data(df)
    lines = [f"Shape: {p['shape']}  |  Duplicate rows: {p['duplicate_rows']}"]
    for col, info in p["columns"].items():
        bits = [f"{col} [{info['role']}]",
                f"nulls={info['null_pct']}%",
                f"unique={info['n_unique']}"]
        if info["flags"]:
            bits.append("flags=" + ",".join(info["flags"]))
        lines.append("  " + " | ".join(bits))
    return "\n".join(lines)


@tool
def inspect_column(column: str) -> str:
    """Inspect a single column in detail. Use when a column's profile shows
    something worth a closer look — high nulls, suspected miscoding, or to see
    its full value distribution. Returns dtype, null count, and value counts.

    Args:
        column: exact name of the column to inspect.
    """
    if column not in df.columns:
        return f"Column '{column}' not found. Available: {list(df.columns)}"
    s = df[column]
    out = [
        f"Column: {column}",
        f"dtype: {s.dtype}",
        f"nulls: {s.isna().sum()} ({s.isna().mean()*100:.1f}%)",
        f"unique: {s.nunique()}",
        f"top values:\n{s.value_counts().head(10).to_string()}",
    ]
    return "\n".join(out)


@tool
def analyze_relationship(col_a: str, col_b: str) -> str:
    """Examine how two columns relate. Use after profiling to test a hypothesis
    about a pair of columns. Picks the right method automatically:
    two numerics -> correlation; one categorical + one numeric -> group means;
    two categoricals -> crosstab.

    Args:
        col_a: first column name.
        col_b: second column name.
    """
    for c in (col_a, col_b):
        if c not in df.columns:
            return f"Column '{c}' not found."
    a, b = df[col_a], df[col_b]
    a_num = pd.api.types.is_numeric_dtype(a)
    b_num = pd.api.types.is_numeric_dtype(b)
    if a_num and b_num:
        r = a.corr(b)
        return f"Correlation between {col_a} and {col_b}: {r:.3f}"
    if a_num != b_num:
        cat, num = (col_b, col_a) if a_num else (col_a, col_b)
        means = df.groupby(cat)[num].mean().sort_values(ascending=False)
        return f"Mean {num} by {cat}:\n{means.to_string()}"
    ct = pd.crosstab(a, b)
    return f"Crosstab {col_a} x {col_b}:\n{ct.to_string()}"


@tool
def check_duplicates() -> str:
    """Investigate duplicate rows. Use when the profile reports duplicates.
    Note: if the dataset has no unique ID column, identical rows may be
    coincidental (different people with the same attributes), not true
    duplicates — do not assume they should be dropped."""
    n = df.duplicated().sum()
    if n == 0:
        return "No duplicate rows found."
    has_id = any(df[c].is_unique for c in df.columns)
    note = ("An ID-like column exists, so duplicates are likely real."
            if has_id else
            "No unique ID column — duplicates may be coincidental, not true repeats.")
    return f"{n} duplicate rows ({n/len(df)*100:.1f}%). {note}"


@tool
def plot(kind: str, columns: list[str], title: str) -> str:
    """Create a chart and save it as a PNG. Use this to visualize a finding
    so your summary can point to evidence, not just numbers.

    Choose 'kind' based on what you want to show:
    - 'hist'    : distribution of ONE numeric column. columns=[col]
    - 'bar'     : a numeric value across categories. columns=[category_col, numeric_col]
    - 'box'     : numeric distribution split by a category. columns=[cat, num]
    - 'scatter' : relationship between TWO numeric columns. columns=[x, y]
    - 'heatmap' : correlation matrix of all numeric columns. columns=[]

    Args:
        kind: one of hist, bar, box, scatter, heatmap.
        columns: column names the chart needs (see above).
        title: short descriptive chart title.
    """
    for c in columns:
        if c not in df.columns:
            return f"Column '{c}' not found."
    fig, ax = plt.subplots(figsize=(7, 4))
    try:
        if kind == "hist":
            df[columns[0]].dropna().hist(ax=ax, bins=30)
        elif kind == "bar":
            df.groupby(columns[0])[columns[1]].mean().plot.bar(ax=ax)
        elif kind == "box":
            sns.boxplot(data=df, x=columns[0], y=columns[1], ax=ax)
        elif kind == "scatter":
            ax.scatter(df[columns[0]], df[columns[1]], alpha=0.4)
            ax.set_xlabel(columns[0]); ax.set_ylabel(columns[1])
        elif kind == "heatmap":
            corr = df.select_dtypes("number").corr()
            sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", ax=ax)
        else:
            plt.close(fig)
            return f"Unknown kind '{kind}'. Use hist, bar, box, scatter, or heatmap."
        ax.set_title(title)
        fig.tight_layout()
        path = f"eda_outputs/{kind}_{'_'.join(columns) or 'corr'}.png"
        fig.savefig(path, dpi=110)
        plt.close(fig)
        return f"Saved chart to {path}"
    except Exception as e:
        plt.close(fig)
        return f"Could not create {kind} chart: {e}"

In [25]:
# ── Agent ──────────────────────────────────────────────────────────
SYSTEM_PROMPT = """You are an EDA analyst. Follow this process, calling tools at each step:

1. Call profile_dataset first.
2. For EVERY column with >50% nulls, call inspect_column on it.
3. Call check_duplicates.
4. Identify the likely target column. Call analyze_relationship between it and
   the 3 most promising predictors — one call per pair.
5. Create at least 3 charts with the plot tool that visualize your key findings.

6. CLEAN the data using handle_missing:
   - Columns with >50% nulls: use 'drop_column'.
   - Numeric columns with nulls: use 'median'.
   - Categorical columns with nulls: use 'mode'.
   Skip columns that have no missing values.

7. Only then write your summary.

Your summary MUST:
- cite specific numbers obtained from tool calls (actual rates, correlations)
- reference the chart files you saved, by path
- include a "Cleaning performed" section listing every change you made
- lead with the single most surprising or actionable finding, not a list

Treat 'maybe_categorical' columns as categories. For relationships between two
such columns, prefer group means / bar charts over correlation."""

agent = create_agent(
    model="openai:gpt-4o",
    tools=[profile_dataset, inspect_column, analyze_relationship,
           check_duplicates, plot, handle_missing],   # ← add it here
    system_prompt=SYSTEM_PROMPT,
)

In [26]:
def run_eda(target_hint=None):
    """Run the EDA agent. Rebind `df` to any DataFrame first to switch datasets."""
    msg = (f"Explore this dataset to understand what predicts {target_hint}."
           if target_hint else
           "Explore this dataset and report the key findings.")
    result = agent.invoke({"messages": [{"role": "user", "content": msg}]})
    print(result["messages"][-1].content)
    return result

In [27]:
result = run_eda(target_hint="survived")

C:\Users\twink\AppData\Local\Temp\ipykernel_58828\1012150917.py:12: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(series, errors="coerce")
C:\Users\twink\AppData\Local\Temp\ipykernel_58828\1012150917.py:12: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(series, errors="coerce")
C:\Users\twink\AppData\Local\Temp\ipykernel_58828\1012150917.py:12: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(series, errors="coerce")
C:\Users\twink\AppData\Local\Temp\ipykernel_58828\1012150917.py:12: UserWarning: Could not 

The most striking finding from the dataset is how gender dramatically influences survival rates. Women had a survival rate of 74.2%, whereas men had only an 18.9% survival rate, as reflected in the bar chart (see "eda_outputs/bar_sex_survived.png"). This suggests gender was a substantial predictor of survival on the Titanic.

Additionally, the passenger class also had a notable impact on survival, showing a correlation of -0.338 with survival status (see "eda_outputs/bar_pclass_survived.png"). The negative correlation indicates that passengers in higher classes were more likely to survive. Meanwhile, age had a minor negative correlation of -0.077 with survival, suggesting younger passengers were slightly more likely to survive (distribution shown in "eda_outputs/hist_age.png").

Cleaning performed included:
- Dropped the 'deck' column due to 77.2% nulls.
- Filled nulls in 'age' with the median value (28.00).
- Filled nulls in 'embarked' and 'embark_town' with their mode ('S' for embark

In [28]:
print(work_df.shape)              # deck dropped → should be 14 cols, not 15
print(work_df.isna().sum())       # age, embarked, embark_town → should be 0

(891, 14)
survived       0
pclass         0
sex            0
age            0
sibsp          0
parch          0
fare           0
embarked       0
class          0
who            0
adult_male     0
embark_town    0
alive          0
alone          0
dtype: int64


In [33]:
%%writefile eda_agent.py
import os
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from langchain.tools import tool
from langchain.agents import create_agent

class EDAAgent:
    """Autonomous EDA + cleaning agent bound to a single dataset."""

    SYSTEM_PROMPT = """You are an EDA analyst. Follow this process, calling tools at each step:

1. Call profile_dataset first.
2. For EVERY column with >50% nulls, call inspect_column on it.
3. Call check_duplicates.
4. Identify the likely target column. Call analyze_relationship between it and
   the 3 most promising predictors — one call per pair.
5. Create at least 3 charts with the plot tool that visualize your key findings.
6. CLEAN the data using handle_missing:
   - Columns with >50% nulls: 'drop_column'.
   - Numeric columns with nulls: 'median'.
   - Categorical columns with nulls: 'mode'.
   Skip columns with no missing values.
7. Only then write your summary.

Your summary MUST:
- cite specific numbers obtained from tool calls
- reference the chart files you saved, by path
- include a "Cleaning performed" section listing every change
- lead with the single most surprising or actionable finding

Treat 'maybe_categorical' columns as categories. For relationships between two
such columns, prefer group means / bar charts over correlation."""

    def __init__(self, df, model="openai:gpt-4o", output_dir="eda_outputs"):
        self.df = df.copy()              # pristine reference
        self.work_df = df.copy()         # mutable working copy
        self.clean_log = []
        self.output_dir = output_dir
        os.makedirs(output_dir, exist_ok=True)
        self.agent = create_agent(
            model=model,
            tools=self._build_tools(),
            system_prompt=self.SYSTEM_PROMPT,
        )

    # ── helpers (plain methods, not tools) ──────────────────────────
    def _infer_role(self, s):
        n = s.notna().sum()
        if s.nunique(dropna=True) <= 1: return "constant"
        if s.nunique(dropna=True) == n: return "id-like"
        if pd.api.types.is_datetime64_any_dtype(s): return "datetime-like"
        if s.dtype == "object":
            try:
                p = pd.to_datetime(s, errors="coerce", format="mixed")
                if p.notna().sum() / n > 0.9: return "datetime-like"
            except (ValueError, TypeError): pass
        if pd.api.types.is_numeric_dtype(s): return "numeric"
        return "categorical"

    def _profile_data(self, df):
        prof = {"shape": list(df.shape),
                "duplicate_rows": int(df.duplicated().sum()),
                "columns": {}}
        for col in df.columns:
            s = df[col]; role = self._infer_role(s); nn = s.dropna()
            nu = int(s.nunique(dropna=True))
            npct = float(round(s.isna().mean() * 100, 1))
            info = {"role": role, "dtype": str(s.dtype), "null_pct": npct,
                    "n_unique": nu, "flags": []}
            if role == "numeric" and len(nn) > 0:
                if pd.api.types.is_bool_dtype(nn): nn = nn.astype(int)
                q1, q3 = nn.quantile(.25), nn.quantile(.75); iqr = q3 - q1
                out = ((nn < q1 - 1.5*iqr) | (nn > q3 + 1.5*iqr)).sum()
                info["stats"] = {"min": float(nn.min()), "max": float(nn.max()),
                                 "mean": round(float(nn.mean()), 2),
                                 "median": float(nn.median()),
                                 "std": round(float(nn.std()), 2),
                                 "outlier_count": int(out)}
            else:
                info["top_values"] = {str(k): int(v) for k, v in nn.value_counts().head(5).items()}
            if npct > 50: info["flags"].append("high_nulls")
            if role == "constant": info["flags"].append("constant")
            if role == "numeric" and nu <= 10: info["flags"].append("maybe_categorical")
            if role == "id-like": info["flags"].append("id_like")
            prof["columns"][col] = info
        return prof

    def _log(self, msg):
        self.clean_log.append(msg)
        return msg

    # ── tools (closures over self) ──────────────────────────────────
    def _build_tools(self):

        @tool
        def profile_dataset() -> str:
            """High-level profile: shape, duplicates, per-column role/nulls/stats.
            Call this FIRST."""
            p = self._profile_data(self.df)
            lines = [f"Shape: {p['shape']} | Duplicate rows: {p['duplicate_rows']}"]
            for col, info in p["columns"].items():
                bits = [f"{col} [{info['role']}]", f"nulls={info['null_pct']}%",
                        f"unique={info['n_unique']}"]
                if info["flags"]: bits.append("flags=" + ",".join(info["flags"]))
                lines.append("  " + " | ".join(bits))
            return "\n".join(lines)

        @tool
        def inspect_column(column: str) -> str:
            """Inspect one column in detail: dtype, nulls, value counts.
            Args: column: exact column name."""
            if column not in self.df.columns:
                return f"Column '{column}' not found. Available: {list(self.df.columns)}"
            s = self.df[column]
            return "\n".join([f"Column: {column}", f"dtype: {s.dtype}",
                f"nulls: {s.isna().sum()} ({s.isna().mean()*100:.1f}%)",
                f"unique: {s.nunique()}",
                f"top values:\n{s.value_counts().head(10).to_string()}"])

        @tool
        def analyze_relationship(col_a: str, col_b: str) -> str:
            """Relate two columns: two numerics->correlation; cat+num->group means;
            two cats->crosstab. Args: col_a, col_b: column names."""
            for c in (col_a, col_b):
                if c not in self.df.columns: return f"Column '{c}' not found."
            a, b = self.df[col_a], self.df[col_b]
            an, bn = pd.api.types.is_numeric_dtype(a), pd.api.types.is_numeric_dtype(b)
            if an and bn: return f"Correlation {col_a}/{col_b}: {a.corr(b):.3f}"
            if an != bn:
                cat, num = (col_b, col_a) if an else (col_a, col_b)
                m = self.df.groupby(cat)[num].mean().sort_values(ascending=False)
                return f"Mean {num} by {cat}:\n{m.to_string()}"
            return f"Crosstab {col_a} x {col_b}:\n{pd.crosstab(a, b).to_string()}"

        @tool
        def check_duplicates() -> str:
            """Investigate duplicate rows. Without a unique ID, duplicates may be
            coincidental — do not assume they should be dropped."""
            n = self.df.duplicated().sum()
            if n == 0: return "No duplicate rows found."
            has_id = any(self.df[c].is_unique for c in self.df.columns)
            note = ("An ID-like column exists, duplicates likely real."
                    if has_id else "No unique ID — duplicates may be coincidental.")
            return f"{n} duplicate rows ({n/len(self.df)*100:.1f}%). {note}"

        @tool
        def plot(kind: str, columns: list[str], title: str) -> str:
            """Save a chart as PNG. kinds: hist[col], bar[cat,num], box[cat,num],
            scatter[x,y], heatmap[]. Args: kind, columns, title."""
            for c in columns:
                if c not in self.df.columns: return f"Column '{c}' not found."
            fig, ax = plt.subplots(figsize=(7, 4))
            try:
                if kind == "hist": self.df[columns[0]].dropna().hist(ax=ax, bins=30)
                elif kind == "bar": self.df.groupby(columns[0])[columns[1]].mean().plot.bar(ax=ax)
                elif kind == "box": sns.boxplot(data=self.df, x=columns[0], y=columns[1], ax=ax)
                elif kind == "scatter":
                    ax.scatter(self.df[columns[0]], self.df[columns[1]], alpha=.4)
                    ax.set_xlabel(columns[0]); ax.set_ylabel(columns[1])
                elif kind == "heatmap":
                    sns.heatmap(self.df.select_dtypes("number").corr(), annot=True,
                                fmt=".2f", cmap="coolwarm", ax=ax)
                else:
                    plt.close(fig); return f"Unknown kind '{kind}'."
                ax.set_title(title); fig.tight_layout()
                path = f"{self.output_dir}/{kind}_{'_'.join(columns) or 'corr'}.png"
                fig.savefig(path, dpi=110); plt.close(fig)
                return f"Saved chart to {path}"
            except Exception as e:
                plt.close(fig); return f"Could not create {kind} chart: {e}"

        @tool
        def handle_missing(column: str, strategy: str) -> str:
            """Handle nulls. strategy: median/mean (numeric only), mode, drop_rows,
            drop_column. Args: column, strategy."""
            if column not in self.work_df.columns:
                return f"Column '{column}' not found. Available: {list(self.work_df.columns)}"
            is_num = pd.api.types.is_numeric_dtype(self.work_df[column])
            nnull = int(self.work_df[column].isna().sum())
            if strategy in ("mean", "median") and not is_num:
                return f"Cannot use '{strategy}' on non-numeric '{column}'. Use 'mode'."
            if nnull == 0 and strategy != "drop_column":
                return f"Column '{column}' has no missing values."
            if strategy == "median":
                v = self.work_df[column].median(); self.work_df[column] = self.work_df[column].fillna(v)
                return self._log(f"Filled {nnull} nulls in '{column}' with median {v:.2f}")
            if strategy == "mean":
                v = self.work_df[column].mean(); self.work_df[column] = self.work_df[column].fillna(v)
                return self._log(f"Filled {nnull} nulls in '{column}' with mean {v:.2f}")
            if strategy == "mode":
                v = self.work_df[column].mode().iloc[0]; self.work_df[column] = self.work_df[column].fillna(v)
                return self._log(f"Filled {nnull} nulls in '{column}' with mode '{v}'")
            if strategy == "drop_rows":
                before = len(self.work_df); self.work_df.dropna(subset=[column], inplace=True)
                return self._log(f"Dropped {before - len(self.work_df)} rows with null '{column}'")
            if strategy == "drop_column":
                self.work_df.drop(columns=[column], inplace=True)
                return self._log(f"Dropped column '{column}'")
            return f"Unknown strategy '{strategy}'."

        return [profile_dataset, inspect_column, analyze_relationship,
                check_duplicates, plot, handle_missing]

    # ── public API ──────────────────────────────────────────────────
    def explore(self, target_hint=None):
        msg = (f"Explore this dataset to understand what predicts {target_hint}."
               if target_hint else "Explore this dataset and report key findings.")
        result = self.agent.invoke({"messages": [{"role": "user", "content": msg}]})
        return result["messages"][-1].content

Writing eda_agent.py


In [34]:
agent = EDAAgent(sns.load_dataset("titanic"))
print(agent.explore(target_hint="survived"))
print("\n--- cleaning log ---")
print(agent.clean_log)
print("\n--- verify ---")
print(agent.work_df.shape, agent.work_df.isna().sum().sum())

### Summary

The most striking finding is the strong influence of **passenger sex** on survival, with females having an average survival rate of approximately 74.2% compared to just 18.9% for males. This is visually confirmed in the chart `eda_outputs/bar_sex_survived.png`. The analysis also highlights:

- The **passenger class** impacts survival, with a negative correlation of -0.338. Higher classes were more likely to survive, as shown in `eda_outputs/box_pclass_survived.png`.
- A positive correlation of 0.257 exists between **fare** and survival, indicating those who paid higher fares had better survival odds. This is depicted in `eda_outputs/scatter_fare_survived.png`.

Our exploration showed no unique identifier to ascertain duplicates, making it hard to remove them definitively, as 12% of records are duplicates potentially by coincidence.

### Cleaning Performed

1. **Dropped Column**: 
   - `deck` due to 77.2% nulls.
   
2. **Filled Nulls**:
   - `age`: Filled 19.9% nulls with t

In [35]:
a1 = EDAAgent(sns.load_dataset("titanic"))
a2 = EDAAgent(sns.load_dataset("tips"))
print(a1.work_df.shape, a2.work_df.shape)   # totally independent

(891, 15) (244, 7)


In [36]:
%%writefile app.py
from fastapi import FastAPI, UploadFile, File, Form
from fastapi.responses import JSONResponse
import pandas as pd
import io
from eda_agent import EDAAgent

app = FastAPI(title="EDA Agent")

@app.get("/")
def health():
    return {"status": "ok"}

@app.post("/analyze")
async def analyze(file: UploadFile = File(...), target: str = Form(None)):
    contents = await file.read()
    df = pd.read_csv(io.BytesIO(contents))
    agent = EDAAgent(df)
    summary = agent.explore(target_hint=target)
    return JSONResponse({
        "summary": summary,
        "cleaning_log": agent.clean_log,
        "rows_after_cleaning": len(agent.work_df),
        "columns_after_cleaning": list(agent.work_df.columns),
    })

Writing app.py


In [37]:
import os
print("eda_agent.py" in os.listdir(), "app.py" in os.listdir())

True True


In [39]:
pip install python-multipart


Note: you may need to restart the kernel to use updated packages.


In [40]:
import os, threading, time, uvicorn
os.environ["OPENAI_API_KEY"] = "Your openai key"   # paste your real key

from app import app   # the FastAPI app from app.py

def run():
    uvicorn.run(app, host="127.0.0.1", port=8000)

threading.Thread(target=run, daemon=True).start()
time.sleep(3)
print("server running → open http://127.0.0.1:8000/docs")

INFO:     Started server process [58828]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


server running → open http://127.0.0.1:8000/docs
INFO:     127.0.0.1:49772 - "GET /docs HTTP/1.1" 200 OK
INFO:     127.0.0.1:49772 - "GET /openapi.json HTTP/1.1" 200 OK
INFO:     127.0.0.1:57019 - "POST /analyze HTTP/1.1" 400 Bad Request
INFO:     127.0.0.1:64904 - "POST /analyze HTTP/1.1" 400 Bad Request
INFO:     127.0.0.1:58568 - "POST /analyze HTTP/1.1" 400 Bad Request
INFO:     127.0.0.1:58568 - "POST /analyze HTTP/1.1" 400 Bad Request
INFO:     127.0.0.1:58568 - "POST /analyze HTTP/1.1" 400 Bad Request
INFO:     127.0.0.1:54196 - "POST /analyze HTTP/1.1" 400 Bad Request


In [ ]:
%%writefile requirements.txt
fastapi
uvicorn[standard]
python-multipart
langchain
langchain-openai
pandas
numpy
seaborn
matplotlib

In [ ]:
%%writefile Dockerfile
FROM python:3.11-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY eda_agent.py app.py ./
EXPOSE 8000
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]

In [ ]:
import os
for f in ["eda_agent.py","app.py","Dockerfile","requirements.txt"]:
    print(f,f in os.listdir())